# 📋 Understand_csv.ipynb — Steam Reviews 2020 第一阶段数据预览

## 项目背景
- **项目**：ECE-9612 小组项目 — 预测游戏评论 Helpfulness
- **数据集**：Steam Reviews 2020（来源：[Kaggle](https://www.kaggle.com/datasets/najzeko/steam-reviews-2020)）

## 本 Notebook 目标
> ⚠️ **这不是完整 EDA，也不是建模 Notebook。**
> 这是**第一阶段预览 Notebook**，专注于：
> 1. 安全地读取超大 CSV，**不爆内存**
> 2. 快速看清数据的基本结构（列名、类型、前几行）
> 3. 粗略分类各列的角色（文本、数值、时间、ID、target候选）
> 4. 为后续更细致的 EDA 和模型选择提供方向性判断

## 执行顺序
请**从上到下逐单元运行**，每个单元尽量可独立运行。  
如果某一步输出很长，可以折叠查看。

---
> 📌 **注意**：若某单元提示内存风险，请先阅读注释再决定是否运行。

## 第 1 节：环境与显示设置

导入所需库，配置 pandas 显示选项，打印当前环境信息。

In [1]:
# ============================================================
# 第 1 节：环境与显示设置
# ============================================================
import sys
import os
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---- 忽略不必要的警告，保持输出干净 ----
warnings.filterwarnings("ignore")

# ---- pandas 显示设置：尽量不截断列名和内容 ----
pd.set_option("display.max_columns", None)       # 显示所有列
pd.set_option("display.max_colwidth", 100)        # 列内容最多显示 100 字符
pd.set_option("display.width", 200)               # 控制台宽度
pd.set_option("display.float_format", "{:.4f}".format)  # 浮点数 4 位小数

# ---- 打印当前环境信息 ----
print("=" * 60)
print("✅ 环境信息")
print("=" * 60)
print(f"Python 版本       : {sys.version}")
print(f"pandas 版本       : {pd.__version__}")
print(f"numpy  版本       : {np.__version__}")
print(f"当前工作目录       : {os.getcwd()}")
print("=" * 60)
print("✅ 环境与显示设置完成，可继续运行下一单元。")

✅ 环境信息
Python 版本       : 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
pandas 版本       : 3.0.1
numpy  版本       : 2.4.3
当前工作目录       : d:\Github_Desktop_File\ECE-9612-Group-Project
✅ 环境与显示设置完成，可继续运行下一单元。


## 第 2 节：设置 CSV 路径与文件检查

> 📌 **请根据你的实际文件名/路径修改 `csv_path` 变量！**  
> 当前默认设置为 `big_reviews.csv`（与本 notebook 同目录）。  
> 如果文件在其他位置，请改为完整路径，例如：  
> `csv_path = r"D:\Data\steam_reviews.csv"`

In [2]:
# ============================================================
# 第 2 节：设置 CSV 路径与文件检查
# ============================================================

# ---- 【重要】请修改这里的路径为你自己的 CSV 文件路径 ----
csv_path = "big_reviews.csv"
# 如果文件不在当前目录，请改成完整路径，例如：
# csv_path = r"D:\Data\big_reviews.csv"

print("=" * 60)
print("📁 CSV 路径检查")
print("=" * 60)
print(f"目标文件路径: {os.path.abspath(csv_path)}")

# ---- 检查文件是否存在 ----
if not os.path.exists(csv_path):
    print()
    print("❌ 错误：找不到指定的 CSV 文件！")
    print(f"   路径: {os.path.abspath(csv_path)}")
    print()
    print("🔧 请检查以下几点：")
    print("   1. 文件名是否拼写正确？")
    print("   2. 文件是否在当前工作目录？")
    print("   3. 或者直接修改 csv_path 变量为完整绝对路径")
    print()
    print("⚠️ 后续单元将无法正常运行，请先修复路径再继续。")
    FILE_EXISTS = False
else:
    # ---- 计算并打印文件大小 ----
    file_size_bytes = os.path.getsize(csv_path)
    file_size_mb = file_size_bytes / (1024 ** 2)
    file_size_gb = file_size_bytes / (1024 ** 3)

    print(f"✅ 文件存在！")
    print(f"   文件大小: {file_size_bytes:,} bytes")
    print(f"            ≈ {file_size_mb:.2f} MB")
    if file_size_gb >= 1:
        print(f"            ≈ {file_size_gb:.2f} GB  ⚠️ 这是一个大文件！")
        print("   💡 提示：本 notebook 将使用 nrows 限制读取量，避免内存溢出。")
    print("=" * 60)
    FILE_EXISTS = True

📁 CSV 路径检查
目标文件路径: d:\Github_Desktop_File\ECE-9612-Group-Project\big_reviews.csv
✅ 文件存在！
   文件大小: 1,553,547,796 bytes
            ≈ 1481.58 MB
            ≈ 1.45 GB  ⚠️ 这是一个大文件！
   💡 提示：本 notebook 将使用 nrows 限制读取量，避免内存溢出。


## 第 3 节：极小量预览（仅读取前 5 行）

> 🎯 **目标**：用最少的内存，第一眼看清楚 CSV 的结构。  
> 只读 5 行，但足以知道：有哪些列、每列是什么类型、数据长什么样。

In [3]:
# ============================================================
# 第 3 节：极小量预览（仅读取前 5 行）
# ============================================================

if not FILE_EXISTS:
    print("⚠️ 文件不存在，跳过此单元。请先修复路径。")
else:
    print("=" * 60)
    print("🔍 极小量预览：只读取前 5 行（内存安全）")
    print("=" * 60)

    try:
        # 只读取前 5 行，极度节省内存
        df_tiny = pd.read_csv(csv_path, nrows=5)

        print(f"✅ 成功读取前 5 行！")
        print(f"   Shape（行 × 列）: {df_tiny.shape}")
        print()

        # ---- 打印列名 ----
        print(f"📋 共有 {df_tiny.shape[1]} 列，列名如下：")
        print("-" * 40)
        for i, col in enumerate(df_tiny.columns, 1):
            print(f"  {i:>3}. {col}")
        print()

        # ---- 打印每列的 dtype ----
        print("📊 每列的数据类型（dtype）：")
        print("-" * 40)
        dtype_df = pd.DataFrame({
            "列名": df_tiny.dtypes.index,
            "dtype": df_tiny.dtypes.values
        })
        print(dtype_df.to_string(index=False))
        print()

        # ---- 打印前 5 行数据 ----
        print("👀 前 5 行数据预览（head）：")
        print("-" * 40)
        print(df_tiny.head())
        print()

        # ---- 第一印象猜测：列类型 ----
        print("🤔 第一印象猜测（基于列名 + dtype）：")
        print("-" * 40)

        # 关键词列表（基于 Steam Reviews 数据集常见列名）
        TEXT_KEYWORDS    = ["review", "text", "comment", "body", "content", "description"]
        NUM_KEYWORDS     = ["votes", "helpful", "funny", "score", "count", "num", "total", "weighted"]
        TIME_KEYWORDS    = ["timestamp", "date", "time", "created", "updated", "posted"]
        BOOL_KEYWORDS    = ["recommend", "steam_purchase", "received_for_free", "written_during"]
        ID_KEYWORDS      = ["id", "appid", "steamid", "author", "userid"]

        guess_text    = []
        guess_num     = []
        guess_time    = []
        guess_bool    = []
        guess_id      = []
        guess_other   = []

        for col in df_tiny.columns:
            col_lower = col.lower()
            dtype_str = str(df_tiny[col].dtype)

            if any(kw in col_lower for kw in TEXT_KEYWORDS) or dtype_str == "object":
                if any(kw in col_lower for kw in TEXT_KEYWORDS):
                    guess_text.append(col)
                elif any(kw in col_lower for kw in TIME_KEYWORDS):
                    guess_time.append(col)
                elif any(kw in col_lower for kw in ID_KEYWORDS):
                    guess_id.append(col)
                elif any(kw in col_lower for kw in BOOL_KEYWORDS):
                    guess_bool.append(col)
                else:
                    guess_other.append(col)
            elif "int" in dtype_str or "float" in dtype_str:
                if any(kw in col_lower for kw in TIME_KEYWORDS):
                    guess_time.append(col)
                elif any(kw in col_lower for kw in ID_KEYWORDS):
                    guess_id.append(col)
                elif any(kw in col_lower for kw in BOOL_KEYWORDS):
                    guess_bool.append(col)
                else:
                    guess_num.append(col)
            elif "bool" in dtype_str:
                guess_bool.append(col)
            else:
                guess_other.append(col)

        # 把猜测结果存到全局变量，后面单元会用到
        GUESS = {
            "text":  guess_text,
            "num":   guess_num,
            "time":  guess_time,
            "bool":  guess_bool,
            "id":    guess_id,
            "other": guess_other,
        }

        print(f"  📝 疑似文本列    : {guess_text if guess_text else '（未识别到）'}")
        print(f"  🔢 疑似数值列    : {guess_num  if guess_num  else '（未识别到）'}")
        print(f"  🕐 疑似时间列    : {guess_time if guess_time else '（未识别到）'}")
        print(f"  ✅ 疑似布尔/类别 : {guess_bool if guess_bool else '（未识别到）'}")
        print(f"  🆔 疑似 ID 列    : {guess_id   if guess_id   else '（未识别到）'}")
        print(f"  ❓ 其他列        : {guess_other if guess_other else '（无）'}")
        print()
        print("💡 注：以上为基于列名+dtype 的粗略猜测，后续单元将进一步细化。")
        print("=" * 60)

    except Exception as e:
        print(f"❌ 读取失败，错误信息：{e}")
        GUESS = {"text": [], "num": [], "time": [], "bool": [], "id": [], "other": []}

🔍 极小量预览：只读取前 5 行（内存安全）
✅ 成功读取前 5 行！
   Shape（行 × 列）: (5, 22)

📋 共有 22 列，列名如下：
----------------------------------------
    1. Unnamed: 0
    2. appid
    3. recommendationid
    4. language
    5. review
    6. timestamp_created
    7. timestamp_updated
    8. voted_up
    9. votes_up
   10. votes_funny
   11. weighted_vote_score
   12. comment_count
   13. steam_purchase
   14. received_for_free
   15. written_during_early_access
   16. steamid
   17. num_games_owned
   18. num_reviews
   19. playtime_forever
   20. playtime_last_two_weeks
   21. playtime_at_review
   22. last_played

📊 每列的数据类型（dtype）：
----------------------------------------
                         列名   dtype
                 Unnamed: 0   int64
                      appid   int64
           recommendationid   int64
                   language     str
                     review     str
          timestamp_created   int64
          timestamp_updated   int64
                   voted_up    bool
                   votes

## 第 4 节：中等规模 Sample 读取与基础统计

> ⚠️ **内存提示**：本节读取 `SAMPLE_ROWS` 行（默认 50,000）。  
> 若机器内存较小（< 8 GB），可把 `SAMPLE_ROWS` 改为 `10000`。  
> **不要改成全量读取**，除非你确认内存足够。

In [4]:
# ============================================================
# 第 4 节：中等规模 Sample 读取与基础统计
# ============================================================

# ---- 【可调整】Sample 行数，默认 50000 行 ----
# 如果内存较小，改成 10000；如果内存宽裕，可以改成 100000
SAMPLE_ROWS = 50000

if not FILE_EXISTS:
    print("⚠️ 文件不存在，跳过此单元。")
else:
    print("=" * 60)
    print(f"📥 读取前 {SAMPLE_ROWS:,} 行数据（Sample 模式）...")
    print("=" * 60)

    try:
        df = pd.read_csv(csv_path, nrows=SAMPLE_ROWS)
        print(f"✅ 成功读取！实际行数: {df.shape[0]:,}，列数: {df.shape[1]}")
        print()

        # ---- 1. Shape ----
        print("─" * 60)
        print(f"📐 Shape（行 × 列）: {df.shape}")
        print()

        # ---- 2. head() ----
        print("─" * 60)
        print("👀 前 3 行（head）：")
        print(df.head(3))
        print()

        # ---- 3. sample(3) ----
        print("─" * 60)
        print("🎲 随机 3 行（sample）：")
        print(df.sample(3, random_state=42))
        print()

        # ---- 4. info() ----
        print("─" * 60)
        print("ℹ️  基本信息（info）：")
        df.info()
        print()

        # ---- 5. 缺失值统计（降序，显示前 15） ----
        print("─" * 60)
        print("🕳️  缺失值统计（按缺失数量降序，前 15 列）：")
        missing = df.isnull().sum()
        missing_pct = (missing / len(df) * 100).round(2)
        missing_df = pd.DataFrame({
            "缺失数量": missing,
            "缺失比例(%)": missing_pct
        }).sort_values("缺失数量", ascending=False)
        missing_df = missing_df[missing_df["缺失数量"] > 0].head(15)
        if missing_df.empty:
            print("  🎉 在 sample 中未发现缺失值！")
        else:
            print(missing_df)
        print()

        # ---- 6. describe() 数值列 ----
        print("─" * 60)
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if num_cols:
            print(f"📊 数值列统计（describe），共 {len(num_cols)} 个数值列：")
            print(df[num_cols].describe())
        else:
            print("📊 未发现数值列。")
        print()

        # ---- 7. describe(include='object') 文本/类别列 ----
        print("─" * 60)
        obj_cols = df.select_dtypes(include=["object"]).columns.tolist()
        if obj_cols:
            print(f"📝 字符串/类别列统计（describe），共 {len(obj_cols)} 个对象列：")
            print(df[obj_cols].describe())
        else:
            print("📝 未发现字符串/类别列。")
        print()

        # ---- 8. 重复行数量 ----
        print("─" * 60)
        dup_count = df.duplicated().sum()
        print(f"🔁 重复行数量: {dup_count:,}（占 sample 的 {dup_count/len(df)*100:.2f}%）")
        print()

        # ---- 9. 每列唯一值数量（排序后前 20） ----
        print("─" * 60)
        print("🔑 每列唯一值数量（按唯一值数量升序，前 20 列）：")
        nunique = df.nunique().sort_values()
        nunique_df = pd.DataFrame({
            "列名": nunique.index,
            "唯一值数量": nunique.values,
            "dtype": [str(df[c].dtype) for c in nunique.index]
        }).head(20)
        print(nunique_df.to_string(index=False))
        print()

        print("=" * 60)
        print("✅ 第 4 节基础统计完成。")

    except MemoryError:
        print("❌ 内存不足！请把 SAMPLE_ROWS 改小（如 10000）再重新运行。")
        df = None
    except Exception as e:
        print(f"❌ 读取失败，错误信息：{e}")
        df = None

📥 读取前 50,000 行数据（Sample 模式）...
✅ 成功读取！实际行数: 50,000，列数: 22

────────────────────────────────────────────────────────────
📐 Shape（行 × 列）: (50000, 22)

────────────────────────────────────────────────────────────
👀 前 3 行（head）：
   Unnamed: 0  appid  recommendationid language                                                               review  timestamp_created  timestamp_updated  voted_up  votes_up  votes_funny  \
0           0    620          81941464  english                                                                  nic         1607579462         1607579462      True         0            0   
1           1    620          81937956  english  bad robot say i has too many fat :(\npussle make me happy after sad         1607576501         1607576501      True         0            0   
2           2    620          81935165  english                                        fun puzzle solving platformer         1607574294         1607574294      True         0            0   

   weighte

## 第 5 节：自动粗分类列类型

> 🎯 **目标**：基于列名关键词 + dtype，对所有列做粗略角色分类。  
> 这不是 100% 精确的分析，只是快速给每列贴一个"初始标签"，供后续参考。

In [5]:
# ============================================================
# 第 5 节：自动粗分类列类型
# ============================================================

if df is None:
    print("⚠️ df 未加载，跳过此单元。请先运行第 4 节。")
else:
    print("=" * 60)
    print("🏷️  自动粗分类列类型（基于列名精确匹配）")
    print("⚠️  注：以下为方向性猜测，非最终结论！")
    print("=" * 60)
    print()

    # ----------------------------------------------------------------
    # Steam Reviews 2020 数据集的已知列（精确集合匹配，速度极快）：
    #   recommendationid, appid, steamid, language, review,
    #   timestamp_created, timestamp_updated, voted_up,
    #   votes_up, votes_funny, weighted_vote_score,
    #   comment_count, steam_purchase, received_for_free,
    #   written_during_early_access, num_games_owned, num_reviews,
    #   playtime_forever, playtime_last_two_weeks, playtime_at_review,
    #   last_played, Unnamed: 0
    # ----------------------------------------------------------------

    # 各类别列名集合（全部用精确匹配，不调用 nunique，速度极快）
    SET_HELPFULNESS = {"votes_up", "votes_funny", "weighted_vote_score", "voted_up"}
    SET_TEXT        = {"review"}
    SET_TIME        = {"timestamp_created", "timestamp_updated"}
    SET_PLAYTIME    = {"playtime_forever", "playtime_last_two_weeks",
                       "playtime_at_review", "last_played"}
    SET_BOOL_CAT    = {"steam_purchase", "received_for_free",
                       "written_during_early_access", "language"}
    SET_ID          = {"recommendationid", "appid", "steamid"}
    SET_DROP        = {"unnamed: 0"}   # 无意义的行索引列

    # ---- 分类函数（纯集合查找，O(1)，不扫描列内容）----
    def classify_columns(dataframe):
        cats = {
            "helpfulness": [],
            "text":        [],
            "time":        [],
            "playtime":    [],
            "bool_cat":    [],
            "numeric":     [],
            "id":          [],
            "drop":        [],
            "other":       [],
        }

        for col in dataframe.columns:
            col_lower = col.lower()
            dtype_str = str(dataframe[col].dtype)

            if col_lower in SET_DROP:
                cats["drop"].append(col)
            elif col_lower in SET_HELPFULNESS:
                cats["helpfulness"].append(col)
            elif col_lower in SET_TEXT:
                cats["text"].append(col)
            elif col_lower in SET_TIME:
                cats["time"].append(col)
            elif col_lower in SET_PLAYTIME:
                cats["playtime"].append(col)
            elif col_lower in SET_BOOL_CAT:
                cats["bool_cat"].append(col)
            elif dtype_str == "bool":
                cats["bool_cat"].append(col)
            elif col_lower in SET_ID:
                cats["id"].append(col)
            elif "int" in dtype_str or "float" in dtype_str:
                cats["numeric"].append(col)
            elif dtype_str in ("object", "str"):
                cats["text"].append(col)   # 剩余 object 列归为疑似文本
            else:
                cats["other"].append(col)

        return cats

    # ---- 执行分类（速度极快，无 nunique 调用）----
    col_categories = classify_columns(df)

    # ---- 预先一次性计算所有列的 nunique（只算一次，缓存结果）----
    print("⏳ 计算各列唯一值数量（只算一次）...")
    nunique_cache = {}
    for col in df.columns:
        dtype_str = str(df[col].dtype)
        # 对长文本 object 列，用 head(5000) 估算，避免全量扫描
        if dtype_str in ("object", "str") and col.lower() in SET_TEXT:
            nunique_cache[col] = df[col].head(5000).nunique()
        else:
            nunique_cache[col] = df[col].nunique()
    print("✅ 唯一值计算完成。")
    print()

    # ---- 打印结果 ----
    label_map = {
        "helpfulness": "🎯 Helpfulness / Votes 核心列",
        "text":        "📝 评论文本列",
        "time":        "🕐 Unix 时间戳列",
        "playtime":    "⏱️  游戏时长列（单位：分钟，非时间戳）",
        "bool_cat":    "✅ 布尔 / 类别列",
        "numeric":     "🔢 其他数值列",
        "id":          "🆔 ID 列",
        "drop":        "🗑️  建议删除列（无意义索引）",
        "other":       "❓ 其他未分类列",
    }

    for key, label in label_map.items():
        cols = col_categories.get(key, [])
        print(f"{label}")
        if cols:
            for c in cols:
                dtype_str = str(df[c].dtype)
                nuniq     = nunique_cache.get(c, "?")
                if "int" in dtype_str or "float" in dtype_str:
                    col_min = df[c].min()
                    col_max = df[c].max()
                    print(f"    · {c:<40} dtype={dtype_str:<10} "
                          f"唯一值≈{nuniq:<6} 范围=[{col_min}, {col_max}]")
                else:
                    print(f"    · {c:<40} dtype={dtype_str:<10} 唯一值≈{nuniq}")
        else:
            print("    （未识别到）")
        print()

    print("=" * 60)
    print("✅ 第 5 节粗分类完成（精确匹配，无误分类，速度快）。")
    print("   col_categories 变量已保存，后续单元将使用。")
    print("=" * 60)


🏷️  自动粗分类列类型（基于列名精确匹配）
⚠️  注：以下为方向性猜测，非最终结论！

⏳ 计算各列唯一值数量（只算一次）...
✅ 唯一值计算完成。

🎯 Helpfulness / Votes 核心列
    · voted_up                                 dtype=bool       唯一值≈2
    · votes_up                                 dtype=int64      唯一值≈88     范围=[0, 297]
    · votes_funny                              dtype=int64      唯一值≈65     范围=[0, 477]
    · weighted_vote_score                      dtype=float64    唯一值≈1163   范围=[0.0, 0.9520151615142822]

📝 评论文本列
    · review                                   dtype=str        唯一值≈4524

🕐 Unix 时间戳列
    · timestamp_created                        dtype=int64      唯一值≈49950  范围=[1580670666, 1607579462]
    · timestamp_updated                        dtype=int64      唯一值≈49951  范围=[1580670666, 1607579462]

⏱️  游戏时长列（单位：分钟，非时间戳）
    · playtime_forever                         dtype=int64      唯一值≈7416   范围=[5, 1942435]
    · playtime_last_two_weeks                  dtype=int64      唯一值≈1322   范围=[0, 20126]
    · playtime_at_review                   

## 第 6 节：文本列预览

> 🎯 **目标**：对识别出的文本列，查看前几条样本 + 长度统计。  
> 仅基于 sample 数据，不做全量处理。

In [6]:
# ============================================================
# 第 6 节：文本列预览
# ============================================================

# 截断长度：每条样本最多显示多少字符
MAX_DISPLAY_CHARS = 200

if df is None:
    print("⚠️ df 未加载，跳过此单元。")
else:
    print("=" * 60)
    print("📝 文本列预览（仅基于 sample 数据）")
    print("=" * 60)

    text_cols = col_categories.get("text", [])

    if not text_cols:
        print()
        print("⚠️ 当前未明显识别到文本列，请手动检查列名。")
        print("   建议检查：所有 object 类型的列，以及 dtype=object 但唯一值很多的列。")
        print()
        # 作为补充，列出所有 object 列
        obj_cols = df.select_dtypes(include=["object"]).columns.tolist()
        print(f"   当前所有 object 列：{obj_cols}")
    else:
        print(f"✅ 识别到 {len(text_cols)} 个疑似文本列：{text_cols}")
        print()

        for col in text_cols:
            print("─" * 60)
            print(f"📌 列名：【{col}】")
            print(f"   dtype: {df[col].dtype}  |  非空行数: {df[col].notna().sum():,}")
            print()

            try:
                # 取前 3 条非空样本
                samples = df[col].dropna().head(3).tolist()
                print("   🔍 前 3 条样本（截断至 200 字符）：")
                for i, s in enumerate(samples, 1):
                    s_str = str(s)
                    truncated = s_str[:MAX_DISPLAY_CHARS] + ("..." if len(s_str) > MAX_DISPLAY_CHARS else "")
                    print(f"   [{i}] {truncated}")
                print()

                # 计算文本长度（字符数）
                lengths = df[col].dropna().astype(str).str.len()
                print("   📏 文本长度统计（字符数）：")
                print(f"       平均长度   : {lengths.mean():.1f} 字符")
                print(f"       中位数长度 : {lengths.median():.1f} 字符")
                print(f"       最长文本   : {lengths.max()} 字符")
                print(f"       最短文本   : {lengths.min()} 字符")
                print(f"       标准差     : {lengths.std():.1f} 字符")

                # 简单判断：是不是真正的"长文本"列
                if lengths.median() > 50:
                    print("   ✅ 判断：中位长度 > 50 字符，很可能是真正的评论文本列。")
                else:
                    print("   ⚠️ 判断：中位长度较短，可能是短字符串/类别值，非典型长文本。")

            except Exception as e:
                print(f"   ❌ 该列预览出错：{e}")

            print()

    print("=" * 60)
    print("✅ 第 6 节文本列预览完成。")

📝 文本列预览（仅基于 sample 数据）
✅ 识别到 1 个疑似文本列：['review']

────────────────────────────────────────────────────────────
📌 列名：【review】
   dtype: str  |  非空行数: 49,813

   🔍 前 3 条样本（截断至 200 字符）：
   [1] nic
   [2] bad robot say i has too many fat :(
pussle make me happy after sad
   [3] fun puzzle solving platformer

   📏 文本长度统计（字符数）：
       平均长度   : 85.2 字符
       中位数长度 : 29.0 字符
       最长文本   : 7999 字符
       最短文本   : 1 字符
       标准差     : 227.1 字符
   ⚠️ 判断：中位长度较短，可能是短字符串/类别值，非典型长文本。

✅ 第 6 节文本列预览完成。


## 第 7 节：数值列预览

> 🎯 **目标**：轻量级查看数值列的基本统计、缺失率、零值比例。  
> 重点关注是否有 votes/helpfulness 风格的计数列。  
> 仅基于 sample，不做完整偏度分析。

In [7]:

# ============================================================
# 第 7 节：数值列预览（轻量级，批量统计无重复扫描）
# ============================================================

if df is None:
    print("⚠️ df 未加载，跳过此单元。")
else:
    print("=" * 60)
    print("🔢 数值列轻量级预览（仅基于 sample 数据）")
    print("=" * 60)

    help_cols     = col_categories.get("helpfulness", [])
    num_cols      = col_categories.get("numeric", [])
    playtime_cols = col_categories.get("playtime", [])

    # 合并去重，只保留真正的数值列（排除 bool 列，bool 不进 describe 的 count 行）
    all_num_cols = list(dict.fromkeys(help_cols + num_cols + playtime_cols))
    all_num_cols = [c for c in all_num_cols if c in df.columns
                    and pd.api.types.is_numeric_dtype(df[c])
                    and not pd.api.types.is_bool_dtype(df[c])]

    if not all_num_cols:
        print("⚠️ 未识别到数值列。")
    else:
        print(f"✅ 共找到 {len(all_num_cols)} 个数值列：")
        print(f"   {all_num_cols}")
        print()

        # ---- 一次性批量计算所有统计量（比逐列循环快很多）----
        sub      = df[all_num_cols]
        desc     = sub.describe()                   # count/mean/std/min/25%/50%/75%/max
        zero_cnt = (sub == 0).sum()                 # 零值数量

        # ---- 构建汇总表 ----
        print("─" * 60)
        print("📊 数值列汇总统计：")

        summary_rows = []
        help_set     = set(help_cols)
        playtime_set = set(playtime_cols)

        for col in all_num_cols:
            total    = len(df)
            notna    = int(desc.loc["count", col])
            miss_pct = (total - notna) / total * 100 if total > 0 else 0
            zero_pct = zero_cnt[col] / notna * 100 if notna > 0 else 0

            # 判断列角色
            if col in help_set:
                role_hint = "helpfulness"
            elif col in playtime_set:
                role_hint = "游戏时长(分钟)"
            elif pd.api.types.is_integer_dtype(df[col]) \
                    and desc.loc["min", col] >= 0 and desc.loc["max", col] < 1e6:
                role_hint = "计数类"
            else:
                role_hint = "连续值"

            summary_rows.append({
                "列名":        col,
                "dtype":      str(df[col].dtype),
                "均值":        round(desc.loc["mean", col], 2),
                "中位数":      round(desc.loc["50%",  col], 2),
                "最大值":      desc.loc["max",  col],
                "最小值":      desc.loc["min",  col],
                "缺失率(%)":   round(miss_pct, 2),
                "零值比例(%)": round(zero_pct, 2),
                "角色":        role_hint,
            })

        summary_df = pd.DataFrame(summary_rows)
        print(summary_df.to_string(index=False))
        print()

        # ---- 重点提示 ----
        print("─" * 60)
        print("💡 重点关注提示：")

        help_num   = [r["列名"] for r in summary_rows if r["角色"] == "helpfulness"]
        count_cols = [r["列名"] for r in summary_rows if r["角色"] == "计数类"]
        pt_cols    = [r["列名"] for r in summary_rows if r["角色"] == "游戏时长(分钟)"]
        high_zero  = [r["列名"] for r in summary_rows
                      if isinstance(r["零值比例(%)"], (int, float)) and r["零值比例(%)"] > 80]

        print(f"   🎯 Helpfulness 核心列  : {help_num if help_num else '（无）'}")
        print(f"      → 这些列是建模 target 或重要特征的首选候选")
        print()
        print(f"   🔢 计数类变量          : {count_cols if count_cols else '（无）'}")
        print(f"      → 非负整数，可能是用户行为统计")
        print()
        print(f"   ⏱️  游戏时长列（分钟）  : {pt_cols if pt_cols else '（无）'}")
        print(f"      → 单位是分钟，不是时间戳！可能是重要的用户行为特征")
        print(f"      → 注意：last_played 也是 Unix timestamp，需单独判断")
        print()
        print(f"   ⚠️  零值比例 > 80% 的列 : {high_zero if high_zero else '（无）'}")
        print(f"      → 零值很多，可能是稀疏特征（如大多数评论无 funny votes）")

    print()
    print("=" * 60)
    print("✅ 第 7 节数值列预览完成。")


🔢 数值列轻量级预览（仅基于 sample 数据）
✅ 共找到 10 个数值列：
   ['votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'num_games_owned', 'num_reviews', 'playtime_forever', 'playtime_last_two_weeks', 'playtime_at_review', 'last_played']

────────────────────────────────────────────────────────────
📊 数值列汇总统计：
                     列名   dtype              均值             中位数             最大值        最小值  缺失率(%)  零值比例(%)          角色
               votes_up   int64          0.3700          0.0000        297.0000     0.0000  0.0000  87.2800 helpfulness
            votes_funny   int64          0.1600          0.0000        477.0000     0.0000  0.0000  95.2700 helpfulness
    weighted_vote_score float64          0.0700          0.0000          0.9520     0.0000  0.0000  85.9900 helpfulness
          comment_count   int64          0.0200          0.0000         30.0000     0.0000  0.0000  98.6600         计数类
        num_games_owned   int64         58.6600         30.0000       5535.0000     0.0000  0.000

## 第 8 节：时间列预览

> 🎯 **目标**：对疑似时间列尝试解析，确认时间范围。  
> 使用 `try-except` 防止解析失败导致崩溃。  
> Steam Reviews 中时间戳通常是 Unix timestamp（整数秒），也可能是字符串格式。

In [8]:

# ============================================================
# 第 8 节：时间列预览（只转换 min/max 两个值，不做全列转换）
# ============================================================

if df is None:
    print("⚠️ df 未加载，跳过此单元。")
else:
    print("=" * 60)
    print("🕐 时间列预览（仅解析真正的 Unix 时间戳列）")
    print("=" * 60)

    true_time_cols = col_categories.get("time", [])

    # 明确排除游戏时长列（playtime 类），这些不是时间戳
    PLAYTIME_EXCLUDE = {"playtime_forever", "playtime_last_two_weeks",
                        "playtime_at_review", "last_played"}

    # 最终解析的时间戳候选列
    all_time_cols = [c for c in true_time_cols if c not in PLAYTIME_EXCLUDE]

    print(f"✅ 确认为 Unix 时间戳的列：{all_time_cols if all_time_cols else '（见下方补充检查）'}")
    print()

    # ---- 快速解析：只转 min/max，不做全列 pd.to_datetime ----
    def parse_unix_timestamp_fast(col_name):
        """只转换 min/max 两个标量，速度极快"""
        col_series = df[col_name].dropna()
        n = len(col_series)
        print(f"📌 列名：【{col_name}】")
        print(f"   dtype: {df[col_name].dtype}  |  非空行数: {n:,}")

        raw_min = col_series.min()
        raw_max = col_series.max()
        print(f"   原始值范围: [{raw_min}, {raw_max}]")

        try:
            ts_min = pd.to_datetime(int(raw_min), unit='s')
            ts_max = pd.to_datetime(int(raw_max), unit='s')
            span_days = (ts_max - ts_min).days

            # 合理性判断：转换后的年份在 2000-2035 之间才认为是时间戳
            if 2000 <= ts_min.year <= 2035 and 2000 <= ts_max.year <= 2035:
                print(f"   ✅ Unix timestamp（秒）解析成功！")
                print(f"      最早时间 : {ts_min}")
                print(f"      最晚时间 : {ts_max}")
                print(f"      时间跨度 : {span_days} 天")
                print(f"      前 3 条示例（原始值 → 解析后）：")
                for orig in col_series.head(3):
                    conv = pd.to_datetime(int(orig), unit='s')
                    print(f"        {int(orig)} → {conv}")
            else:
                print(f"   ⚠️ 转换结果年份异常（{ts_min.year}~{ts_max.year}），"
                      f"该列可能不是时间戳。")
        except Exception as e:
            print(f"   ❌ 解析失败：{e}")

    for col in all_time_cols:
        print("─" * 60)
        parse_unix_timestamp_fast(col)
        print()

    # ---- 补充：检查 last_played 是否是 Unix timestamp ----
    print("─" * 60)
    print("🔍 补充检查：last_played 是游戏时长还是 Unix 时间戳？")
    if "last_played" in df.columns:
        lp_series = df["last_played"].dropna()
        lp_median = lp_series.median()
        print(f"   last_played 中位数: {lp_median:.0f}")
        # Unix timestamp 2010-2033 范围约为 1.26e9 ~ 2.0e9
        if 1_200_000_000 <= lp_median <= 2_000_000_000:
            print("   → 中位数落在 Unix timestamp 2010-2033 年范围，很可能是 Unix 时间戳！")
            print("   → 含义：用户最后一次玩这个游戏的时间")
            print()
            parse_unix_timestamp_fast("last_played")
        else:
            print(f"   → 中位数不在 Unix timestamp 范围，可能是其他数值（如游戏时长分钟）。")
    else:
        print("   last_played 列不存在。")

    # ---- 明确说明游戏时长列不是时间戳 ----
    print()
    print("─" * 60)
    print("⏱️  以下列是【游戏时长（分钟）】，不是时间戳，不做时间解析：")
    playtime_cols = col_categories.get("playtime", [])
    exclude_display = [c for c in playtime_cols if c != "last_played"]
    if exclude_display:
        # 一次性批量取中位数（describe 已在第7节算过，这里直接 median()，50k 行很快）
        for col in exclude_display:
            if col in df.columns:
                med = df[col].median()          # 单列 median，50k 行 <1ms
                print(f"   · {col:<35} 中位数={med:.0f} 分钟（≈ {med/60:.1f} 小时）")
        print("   💡 这些列表示玩家的游戏时长，可作为用户行为特征使用，不需要转时间格式。")
    else:
        print("   （无）")

    print()
    print("=" * 60)
    print("✅ 第 8 节时间列预览完成。")


🕐 时间列预览（仅解析真正的 Unix 时间戳列）
✅ 确认为 Unix 时间戳的列：['timestamp_created', 'timestamp_updated']

────────────────────────────────────────────────────────────
📌 列名：【timestamp_created】
   dtype: int64  |  非空行数: 50,000
   原始值范围: [1580670666, 1607579462]
   ✅ Unix timestamp（秒）解析成功！
      最早时间 : 2020-02-02 19:11:06
      最晚时间 : 2020-12-10 05:51:02
      时间跨度 : 311 天
      前 3 条示例（原始值 → 解析后）：
        1607579462 → 2020-12-10 05:51:02
        1607576501 → 2020-12-10 05:01:41
        1607574294 → 2020-12-10 04:24:54

────────────────────────────────────────────────────────────
📌 列名：【timestamp_updated】
   dtype: int64  |  非空行数: 50,000
   原始值范围: [1580670666, 1607579462]
   ✅ Unix timestamp（秒）解析成功！
      最早时间 : 2020-02-02 19:11:06
      最晚时间 : 2020-12-10 05:51:02
      时间跨度 : 311 天
      前 3 条示例（原始值 → 解析后）：
        1607579462 → 2020-12-10 05:51:02
        1607576501 → 2020-12-10 05:01:41
        1607574294 → 2020-12-10 04:24:54

────────────────────────────────────────────────────────────
🔍 补充检查：last_played

## 第 9 节：第一阶段总结

> ✅ 这是整个预览 Notebook 的收尾部分。  
> 以下总结**完全基于预览阶段的观察**，是**方向性判断，不是最终结论**。  
> 随着后续详细 EDA 的推进，这些判断可能会被修正。

---

### 总结框架

我们将回答以下 6 个关键问题：

| # | 问题 |
|---|------|
| 1 | CSV 大概由哪些类型的列组成？ |
| 2 | 哪些列最可能和 review helpfulness 相关？ |
| 3 | 哪些列最可能是文本主特征？ |
| 4 | 哪些列需要后续重点分析？ |
| 5 | 数据更适合哪类建模方向？ |
| 6 | 下一步应该优先做哪些分析？ |

In [9]:

# ============================================================
# 第 9 节：第一阶段总结（复用已有缓存，不重复计算）
# ============================================================

print("=" * 70)
print("📋  第一阶段预览总结报告")
print("    数据集：Steam Reviews 2020  |  文件：big_reviews.csv")
print("    全量总行数：4,374,931 行  |  Sample：50,000 行（覆盖 1.14%）")
print("    ⚠️  以下均为方向性判断，非最终结论")
print("=" * 70)

if df is None:
    print("⚠️ df 未加载，无法生成详细总结。请先运行前面的单元。")
else:
    # ---- Q1：CSV 由哪些类型的列组成 ----
    print()
    print("─" * 70)
    print("❓ Q1：这个 CSV 大概由哪些类型的列组成？")
    print("─" * 70)
    print(f"   总列数：{df.shape[1]} 列，Sample 行数：{df.shape[0]:,} 行")
    print()

    for key, label in [
        ("helpfulness", "🎯 Helpfulness/Votes 相关列"),
        ("text",        "📝 评论文本列"),
        ("time",        "🕐 Unix 时间戳列"),
        ("bool_cat",    "✅ 布尔 / 类别列"),
        ("playtime",    "⏱️  游戏时长列（分钟，非时间戳）"),
        ("numeric",     "🔢 其他数值列"),
        ("id",          "🆔 ID 列"),
        ("other",       "❓ 其他列"),
    ]:
        cols  = col_categories.get(key, [])
        names = ", ".join(cols)
        print(f"   {label}：{len(cols)} 列  →  {names if cols else '（未识别）'}")

    # ---- Q2：Helpfulness 相关列（复用 nunique_cache，不重复调用 nunique）----
    print()
    print("─" * 70)
    print("❓ Q2：哪些列最可能和 review helpfulness 相关？")
    print("─" * 70)
    help_cols = col_categories.get("helpfulness", [])
    if help_cols:
        print(f"   识别到 {len(help_cols)} 个候选列：")
        for col in help_cols:
            dtype_str  = str(df[col].dtype)
            nuniq      = nunique_cache.get(col, "?")   # 直接读缓存，无重复计算
            sample_vals = df[col].dropna().head(3).tolist()
            print(f"   · {col:<45} dtype={dtype_str:<10} 唯一值={nuniq}  样本={sample_vals}")
        print()
        print("   💡 分析建议：")
        print("      · votes_up            → 回归 target：获得多少 helpful votes")
        print("      · weighted_vote_score → Steam 官方加权分，值域 [0,1]，可作为回归 target")
        print("      · voted_up            → True/False，是否推荐游戏，可作为分类 target")
        print("      · votes_funny         → 搞笑票数，可作为辅助特征")
    else:
        print("   ⚠️ 未识别到 helpfulness 相关列，请手动检查。")

    # ---- Q3：文本主特征列（用 describe 批量获取长度统计）----
    print()
    print("─" * 70)
    print("❓ Q3：哪些列最可能是文本主特征？")
    print("─" * 70)
    text_cols = col_categories.get("text", [])
    if text_cols:
        for col in text_cols:
            if df[col].notna().any():
                sample_text = str(df[col].dropna().iloc[0])[:80]
                # 用 describe() 一次拿到所有长度统计（比 .mean() .median() 各调一次快）
                len_desc = df[col].dropna().astype(str).str.len().describe()
                print(f"   · {col}")
                print(f"     样本   : {sample_text}...")
                print(f"     长度   : 平均 {len_desc['mean']:.0f} 字符，"
                      f"中位数 {len_desc['50%']:.0f} 字符")
        print()
        print("   💡 review 列是最核心的 NLP 特征，建议用于：")
        print("      TF-IDF 向量化 / Sentence-BERT 嵌入 / 情感分析")
    else:
        print("   ⚠️ 未识别到文本列，请手动排查 object 类型列。")

    # ---- Q4：需要后续重点分析的列 ----
    print()
    print("─" * 70)
    print("❓ Q4：哪些列需要后续重点分析？")
    print("─" * 70)
    print("   ⭐ 第一优先级（target 候选）：")
    print("      · votes_up             — 回归目标：获得多少 helpful votes")
    print("      · weighted_vote_score  — 回归目标：Steam 加权帮助分（已处理长尾）")
    print("      · voted_up             — 分类目标：是否推荐游戏（正/负向评价）")
    print()
    print("   ⭐ 第二优先级（核心特征）：")
    print("      · review               — 评论文本，NLP 主特征")
    print("      · playtime_forever     — 总游戏时长（分钟），用户投入程度")
    print("      · playtime_at_review   — 写评论时的游戏时长，与评论质量高度相关")
    print("      · language             — 评论语言，影响建模策略（是否只用英文？）")
    print()
    print("   ⭐ 第三优先级（辅助特征）：")
    print("      · num_games_owned      — 用户拥有游戏数，衡量用户经验")
    print("      · num_reviews          — 用户写过的评论数，衡量用户活跃度")
    print("      · steam_purchase       — 是否通过 Steam 购买（非赠送）")
    print("      · written_during_early_access — 是否在抢先体验期写的评论")
    print("      · timestamp_created    — 评论创建时间，可做时序分析")
    print()
    print("   ⚠️ 建议排除（不用于建模）：")
    print("      · recommendationid, steamid, appid — 纯 ID 列，无预测价值")
    print("      · Unnamed: 0                       — CSV 行索引列，删除")
    print("      · last_played                      — 含义模糊，需进一步确认")

    # ---- Q5：建模方向 ----
    print()
    print("─" * 70)
    print("❓ Q5：数据更适合哪类建模方向？")
    print("─" * 70)
    print("   基于预览结果，本数据集支持多种建模方向：")
    print()
    print("   ✅ [强烈推荐] 回归预测 helpfulness：")
    print("      目标列：votes_up 或 weighted_vote_score")
    print("      特征：review 文本 + playtime + 用户统计列")
    print("      模型：Ridge / LightGBM / Sentence-BERT + 回归头")
    print()
    print("   ✅ [推荐] 二分类（是否推荐游戏）：")
    print("      目标列：voted_up（True/False）")
    print("      注意：voted_up 可能与 helpfulness 关联但不等同")
    print("      模型：Logistic Regression / XGBoost / BERT")
    print()
    print("   ✅ [推荐] 结构化 + 文本融合：")
    print("      review 文本嵌入 + 结构化特征（playtime, num_games_owned 等）→ 拼接 → 预测")
    print("      这是最可能取得好效果的方向")
    print()
    print("   ⚠️ 注意：数据集共 437 万行，只有 1 个 appid（单款游戏数据），")
    print("           建模时无需做跨游戏泛化。")

    # ---- Q6：下一步建议 ----
    print()
    print("─" * 70)
    print("❓ Q6：下一步详细分析应该优先做哪些？")
    print("─" * 70)
    next_steps = [
        "1.  [紧急] 确认 target 列：votes_up、weighted_vote_score、voted_up 的分布分析",
        "2.  [紧急] 语言过滤：language 列统计各语言比例，决定是否只用英文评论",
        "3.  [重要] votes_up 分布：极度长尾（大多数评论 votes=0），考虑 log 变换或二分类",
        "4.  [重要] 文本预处理：review 列的清洗（空评论、极短评论、非英文过滤）",
        "5.  [重要] 时间分析：timestamp_created 的分布，确认数据是否为 2020 年",
        "6.  [重要] playtime 特征工程：playtime_forever 和 playtime_at_review 的分布与变换",
        "7.  [一般] 样本不平衡：voted_up 正负比例，以及 votes_up=0 的比例",
        "8.  [一般] 相关性分析：votes_up 与 playtime / num_games_owned / text length 的相关性",
        "9.  [一般] 缺失值处理：playtime_at_review 是否有缺失，决定填充策略",
        "10. [备用] 全量建模：确认用 chunk 方式或 Dask 处理全量 437 万行",
    ]
    for step in next_steps:
        print(f"   {step}")

    print()
    print("=" * 70)
    print("✅ 第一阶段预览完成！")
    print("   建议：下一步优先做 votes_up 分布分析 + 语言过滤 + 文本长度分布。")
    print("=" * 70)


📋  第一阶段预览总结报告
    数据集：Steam Reviews 2020  |  文件：big_reviews.csv
    全量总行数：4,374,931 行  |  Sample：50,000 行（覆盖 1.14%）
    ⚠️  以下均为方向性判断，非最终结论

──────────────────────────────────────────────────────────────────────
❓ Q1：这个 CSV 大概由哪些类型的列组成？
──────────────────────────────────────────────────────────────────────
   总列数：22 列，Sample 行数：50,000 行

   🎯 Helpfulness/Votes 相关列：4 列  →  voted_up, votes_up, votes_funny, weighted_vote_score
   📝 评论文本列：1 列  →  review
   🕐 Unix 时间戳列：2 列  →  timestamp_created, timestamp_updated
   ✅ 布尔 / 类别列：4 列  →  language, steam_purchase, received_for_free, written_during_early_access
   ⏱️  游戏时长列（分钟，非时间戳）：4 列  →  playtime_forever, playtime_last_two_weeks, playtime_at_review, last_played
   🔢 其他数值列：3 列  →  comment_count, num_games_owned, num_reviews
   🆔 ID 列：3 列  →  appid, recommendationid, steamid
   ❓ 其他列：0 列  →  （未识别）

──────────────────────────────────────────────────────────────────────
❓ Q2：哪些列最可能和 review helpfulness 相关？
─────────────────────────────────────────

---

## 附录：用 Chunk 方式安全统计全量行数

> ⚠️ **这一步可能耗时较长**（取决于文件大小）。  
> 如果 CSV 非常大（> 2 GB），可以跳过，或者在后台运行。  
> 这里用 chunk 方式逐块扫描，**不会把整个文件读进内存**。

In [10]:
# ============================================================
# 附录：用 Chunk 方式安全统计全量行数
# ⚠️ 这步可能耗时，可选择跳过
# ============================================================

# 每次读取的块大小（行数）
CHUNK_SIZE = 100_000

if not FILE_EXISTS:
    print("⚠️ 文件不存在，跳过此单元。")
else:
    print("=" * 60)
    print(f"🔄 使用 chunk 方式统计全量行数（chunk_size={CHUNK_SIZE:,}）")
    print("   这不会把整个文件读进内存，但需要一点时间...")
    print("=" * 60)

    try:
        total_rows = 0
        chunk_count = 0

        # pd.read_csv 的 chunksize 参数会返回一个迭代器，逐块读取
        reader = pd.read_csv(csv_path, chunksize=CHUNK_SIZE, low_memory=False)

        for chunk in reader:
            total_rows += len(chunk)
            chunk_count += 1
            # 每处理 10 个 chunk 打印一次进度
            if chunk_count % 10 == 0:
                print(f"   已处理 {chunk_count} 个 chunk，当前累计行数：{total_rows:,}")

        print()
        print(f"✅ 全量扫描完成！")
        print(f"   总行数（不含表头）: {total_rows:,}")
        print(f"   处理 chunk 数      : {chunk_count}")
        print(f"   我们的 sample 覆盖率: {SAMPLE_ROWS / total_rows * 100:.2f}%")
        print()
        print(f"💡 提示：全量 {total_rows:,} 行数据，")
        print(f"   若每行约 1KB，全量约占内存 {total_rows * 1e3 / 1e9:.1f} GB。")
        print(f"   建议后续分析始终使用 chunk 或 sample 方式处理。")

    except Exception as e:
        print(f"❌ Chunk 扫描失败：{e}")

🔄 使用 chunk 方式统计全量行数（chunk_size=100,000）
   这不会把整个文件读进内存，但需要一点时间...
   已处理 10 个 chunk，当前累计行数：1,000,000
   已处理 10 个 chunk，当前累计行数：1,000,000
   已处理 20 个 chunk，当前累计行数：2,000,000
   已处理 20 个 chunk，当前累计行数：2,000,000
   已处理 30 个 chunk，当前累计行数：3,000,000
   已处理 30 个 chunk，当前累计行数：3,000,000
   已处理 40 个 chunk，当前累计行数：4,000,000
   已处理 40 个 chunk，当前累计行数：4,000,000

✅ 全量扫描完成！
   总行数（不含表头）: 4,374,931
   处理 chunk 数      : 44
   我们的 sample 覆盖率: 1.14%

💡 提示：全量 4,374,931 行数据，
   若每行约 1KB，全量约占内存 4.4 GB。
   建议后续分析始终使用 chunk 或 sample 方式处理。

✅ 全量扫描完成！
   总行数（不含表头）: 4,374,931
   处理 chunk 数      : 44
   我们的 sample 覆盖率: 1.14%

💡 提示：全量 4,374,931 行数据，
   若每行约 1KB，全量约占内存 4.4 GB。
   建议后续分析始终使用 chunk 或 sample 方式处理。
